In [3]:
import pandas as pd
import numpy as np

df = pd.read_parquet("cleaned_crime_data.parquet")

print("Dataset shape:", df.shape)

df.head()

Dataset shape: (3046607, 21)


,cmplnt_fr_dt,cmplnt_fr_tm,cmplnt_to_dt,cmplnt_to_tm,rpt_dt,ky_cd,ofns_desc,pd_cd,pd_desc,crm_atpt_cptd_cd,...,boro_nm,addr_pct_cd,loc_of_occur_desc,prem_typ_desc,latitude,longitude,year,month,day_of_week,hour
0,2020-01-01,2026-08-21 00:01:00,2021-06-30,2026-08-21 23:59:00,2021-07-02,340,FRAUDS,718.0,"FRAUD,UNCLASSIFIED-MISDEMEANOR",COMPLETED,...,BRONX,47.0,INSIDE,RESIDENCE-HOUSE,40.898612,-73.867384,2020,1,2,0
1,2020-01-01,2026-08-21 08:00:00,2020-02-27,2026-08-21 08:00:00,2020-02-27,112,THEFT-FRAUD,739.0,"FRAUD,UNCLASSIFIED-FELONY",COMPLETED,...,STATEN ISLAND,121.0,INSIDE,RESIDENCE-HOUSE,40.632157,-74.165787,2020,1,2,8
2,2020-01-01,2026-08-21 16:30:00,NaT,NaT,2020-01-01,578,HARRASSMENT 2,637.0,"HARASSMENT,SUBD 1,CIVILIAN",COMPLETED,...,STATEN ISLAND,122.0,INSIDE,RESIDENCE-HOUSE,40.565063,-74.098826,2020,1,2,16
3,2020-01-01,2026-08-21 00:30:00,2020-01-01,2026-08-21 00:33:00,2020-01-01,355,OFFENSES AGAINST THE PERSON,115.0,RECKLESS ENDANGERMENT 2,COMPLETED,...,STATEN ISLAND,122.0,OPPOSITE OF,RESIDENCE-HOUSE,40.553653,-74.149886,2020,1,2,0
4,2020-01-01,2026-08-21 03:00:00,2020-01-01,2026-08-21 03:30:00,2020-01-01,365,ADMINISTRATIVE CODE,877.0,UNLAWFUL DISCLOSURE OF AN INTIMATE IMAGE,COMPLETED,...,BROOKLYN,68.0,INSIDE,RESIDENCE - APT. HOUSE,40.622500,-74.025503,2020,1,2,3


In [5]:
grid_size = 0.01

df["lat_grid"] = df["lat_grid"].round(2)
df["lon_grid"] = df["lon_grid"].round(2)

df["grid_id"] = (
    df["lat_grid"].astype(str) + "_" +
    df["lon_grid"].astype(str)
)

print("Number of geographic grid cells:", df["grid_id"].nunique())

df[["latitude", "longitude", "lat_grid", "lon_grid", "grid_id"]].head()

Number of geographic grid cells: 946


,latitude,longitude,lat_grid,lon_grid,grid_id
0,40.898612,-73.867384,40.89,-73.87,40.89_-73.87
1,40.632157,-74.165787,40.63,-74.17,40.63_-74.17
2,40.565063,-74.098826,40.56,-74.10,40.56_-74.1
3,40.553653,-74.149886,40.55,-74.15,40.55_-74.15
4,40.622500,-74.025503,40.62,-74.03,40.62_-74.03


In [6]:
grid_counts = df["grid_id"].value_counts()

print("Number of geographic grid cells:", len(grid_counts))
print("\nAverage incidents per grid:", round(grid_counts.mean(), 2))
print("Median incidents per grid:", grid_counts.median())
print("Minimum incidents in a grid:", grid_counts.min())
print("Maximum incidents in a grid:", grid_counts.max())

print("\nTop 10 busiest grid cells:")
print(grid_counts.head(10))

Number of geographic grid cells: 946

Average incidents per grid: 3220.47
Median incidents per grid: 1475.0
Minimum incidents in a grid: 1
Maximum incidents in a grid: 40415

Top 10 busiest grid cells:
grid_id
40.75_-73.99    40415
40.75_-74.0     25936
40.86_-73.9     22884
40.81_-73.92    21810
40.79_-73.95    21388
40.76_-73.99    21278
40.85_-73.91    20798
40.83_-73.92    20351
40.8_-73.95     20141
40.72_-74.0     19928
Name: count, dtype: int64


In [7]:
# Time-of-day category
df["time_period"] = pd.cut(
    df["hour"],
    bins=[-1, 5, 11, 17, 21, 23],
    labels=["Night", "Morning", "Afternoon", "Evening", "Late Night"]
)

# Weekend indicator
df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)

df[["hour", "day_of_week", "time_period", "is_weekend"]].head(10)

,hour,day_of_week,time_period,is_weekend
0,0,2,Night,0
1,8,2,Morning,0
2,16,2,Afternoon,0
3,0,2,Night,0
4,3,2,Night,0
5,20,2,Evening,0
6,18,2,Evening,0
7,2,2,Night,0
8,9,2,Morning,0
9,1,2,Night,0


In [8]:
df["grid_crime_count"] = (
    df.groupby("grid_id")["grid_id"]
    .transform("count")
)

df[["grid_id", "grid_crime_count"]].head()

,grid_id,grid_crime_count
0,40.89_-73.87,2469.0
1,40.63_-74.17,3129.0
2,40.56_-74.1,159.0
3,40.55_-74.15,569.0
4,40.62_-74.03,5103.0


In [9]:
df["grid_hour_crime_count"] = (
    df.groupby(["grid_id", "hour"])["grid_id"]
    .transform("count")
)

df[["grid_id", "hour", "grid_hour_crime_count"]].head(10)

,grid_id,hour,grid_hour_crime_count
0,40.89_-73.87,0,148.0
1,40.63_-74.17,8,120.0
2,40.56_-74.1,16,14.0
3,40.55_-74.15,0,29.0
4,40.62_-74.03,3,89.0
5,40.6_-74.1,20,18.0
6,40.57_-74.15,18,3.0
7,40.63_-74.14,2,136.0
8,40.6_-74.08,9,48.0
9,40.63_-74.11,1,36.0


In [10]:
df["grid_day_crime_count"] = (
    df.groupby(["grid_id", "day_of_week"])["grid_id"]
    .transform("count")
)

df[["grid_id", "day_of_week", "grid_day_crime_count"]].head(10)

,grid_id,day_of_week,grid_day_crime_count
0,40.89_-73.87,2,369.0
1,40.63_-74.17,2,460.0
2,40.56_-74.1,2,24.0
3,40.55_-74.15,2,90.0
4,40.62_-74.03,2,759.0
5,40.6_-74.1,2,81.0
6,40.57_-74.15,2,10.0
7,40.63_-74.14,2,572.0
8,40.6_-74.08,2,126.0
9,40.63_-74.11,2,183.0


In [11]:
df["grid_time_period_crime_count"] = (
    df.groupby(["grid_id", "time_period"], observed=True)["grid_id"]
    .transform("count")
)

df[[
    "grid_id",
    "time_period",
    "grid_time_period_crime_count"
]].head(10)

,grid_id,time_period,grid_time_period_crime_count
0,40.89_-73.87,Night,502.0
1,40.63_-74.17,Morning,672.0
2,40.56_-74.1,Afternoon,61.0
3,40.55_-74.15,Night,93.0
4,40.62_-74.03,Night,685.0
5,40.6_-74.1,Evening,109.0
6,40.57_-74.15,Evening,15.0
7,40.63_-74.14,Night,728.0
8,40.6_-74.08,Morning,207.0
9,40.63_-74.11,Night,180.0


In [12]:
df["grid_weekend_crime_count"] = (
    df.groupby(["grid_id", "is_weekend"])["grid_id"]
    .transform("count")
)

df[
    ["grid_id", "is_weekend", "grid_weekend_crime_count"]
].head(10)

,grid_id,is_weekend,grid_weekend_crime_count
0,40.89_-73.87,0,1762.0
1,40.63_-74.17,0,2271.0
2,40.56_-74.1,0,112.0
3,40.55_-74.15,0,430.0
4,40.62_-74.03,0,3803.0
5,40.6_-74.1,0,459.0
6,40.57_-74.15,0,51.0
7,40.63_-74.14,0,2812.0
8,40.6_-74.08,0,576.0
9,40.63_-74.11,0,835.0


In [13]:
max_grid_count = df["grid_crime_count"].max()

df["grid_risk_score"] = (
    df["grid_crime_count"] / max_grid_count
)

df[[
    "grid_id",
    "grid_crime_count",
    "grid_risk_score"
]].head(10)

,grid_id,grid_crime_count,grid_risk_score
0,40.89_-73.87,2469.0,0.061091
1,40.63_-74.17,3129.0,0.077422
2,40.56_-74.1,159.0,0.003934
3,40.55_-74.15,569.0,0.014079
4,40.62_-74.03,5103.0,0.126265
5,40.6_-74.1,624.0,0.015440
6,40.57_-74.15,69.0,0.001707
7,40.63_-74.14,4037.0,0.099889
8,40.6_-74.08,805.0,0.019918
9,40.63_-74.11,1120.0,0.027712


In [14]:
max_grid_hour_count = df["grid_hour_crime_count"].max()

df["grid_hour_risk_score"] = (
    df["grid_hour_crime_count"] / max_grid_hour_count
)

df[[
    "grid_id",
    "hour",
    "grid_hour_crime_count",
    "grid_hour_risk_score"
]].head(10)

,grid_id,hour,grid_hour_crime_count,grid_hour_risk_score
0,40.89_-73.87,0,148.0,0.049865
1,40.63_-74.17,8,120.0,0.040431
2,40.56_-74.1,16,14.0,0.004717
3,40.55_-74.15,0,29.0,0.009771
4,40.62_-74.03,3,89.0,0.029987
5,40.6_-74.1,20,18.0,0.006065
6,40.57_-74.15,18,3.0,0.001011
7,40.63_-74.14,2,136.0,0.045822
8,40.6_-74.08,9,48.0,0.016173
9,40.63_-74.11,1,36.0,0.012129


In [15]:
max_grid_day_count = df["grid_day_crime_count"].max()

df["grid_day_risk_score"] = (
    df["grid_day_crime_count"] / max_grid_day_count
)

df[[
    "grid_id",
    "day_of_week",
    "grid_day_crime_count",
    "grid_day_risk_score"
]].head(10)

,grid_id,day_of_week,grid_day_crime_count,grid_day_risk_score
0,40.89_-73.87,2,369.0,0.057378
1,40.63_-74.17,2,460.0,0.071529
2,40.56_-74.1,2,24.0,0.003732
3,40.55_-74.15,2,90.0,0.013995
4,40.62_-74.03,2,759.0,0.118022
5,40.6_-74.1,2,81.0,0.012595
6,40.57_-74.15,2,10.0,0.001555
7,40.63_-74.14,2,572.0,0.088944
8,40.6_-74.08,2,126.0,0.019593
9,40.63_-74.11,2,183.0,0.028456


In [16]:
max_grid_time_period_count = df["grid_time_period_crime_count"].max()

df["grid_time_period_risk_score"] = (
    df["grid_time_period_crime_count"] / max_grid_time_period_count
)

df[[
    "grid_id",
    "time_period",
    "grid_time_period_crime_count",
    "grid_time_period_risk_score"
]].head(10)

,grid_id,time_period,grid_time_period_crime_count,grid_time_period_risk_score
0,40.89_-73.87,Night,502.0,0.031147
1,40.63_-74.17,Morning,672.0,0.041695
2,40.56_-74.1,Afternoon,61.0,0.003785
3,40.55_-74.15,Night,93.0,0.005770
4,40.62_-74.03,Night,685.0,0.042502
5,40.6_-74.1,Evening,109.0,0.006763
6,40.57_-74.15,Evening,15.0,0.000931
7,40.63_-74.14,Night,728.0,0.045170
8,40.6_-74.08,Morning,207.0,0.012844
9,40.63_-74.11,Night,180.0,0.011168


In [17]:
max_grid_weekend_count = df["grid_weekend_crime_count"].max()

df["grid_weekend_risk_score"] = (
    df["grid_weekend_crime_count"] / max_grid_weekend_count
)

df[[
    "grid_id",
    "is_weekend",
    "grid_weekend_crime_count",
    "grid_weekend_risk_score"
]].head(10)

,grid_id,is_weekend,grid_weekend_crime_count,grid_weekend_risk_score
0,40.89_-73.87,0,1762.0,0.059297
1,40.63_-74.17,0,2271.0,0.076426
2,40.56_-74.1,0,112.0,0.003769
3,40.55_-74.15,0,430.0,0.014471
4,40.62_-74.03,0,3803.0,0.127983
5,40.6_-74.1,0,459.0,0.015447
6,40.57_-74.15,0,51.0,0.001716
7,40.63_-74.14,0,2812.0,0.094632
8,40.6_-74.08,0,576.0,0.019384
9,40.63_-74.11,0,835.0,0.028100


In [18]:
new_features = [
    "lat_grid",
    "lon_grid",
    "grid_id",
    "time_period",
    "is_weekend",
    "grid_crime_count",
    "grid_hour_crime_count",
    "grid_day_crime_count",
    "grid_time_period_crime_count",
    "grid_weekend_crime_count",
    "grid_risk_score",
    "grid_hour_risk_score",
    "grid_day_risk_score",
    "grid_time_period_risk_score",
    "grid_weekend_risk_score"
]

print("Number of engineered features:", len(new_features))
print(new_features)

Number of engineered features: 15
['lat_grid', 'lon_grid', 'grid_id', 'time_period', 'is_weekend', 'grid_crime_count', 'grid_hour_crime_count', 'grid_day_crime_count', 'grid_time_period_crime_count', 'grid_weekend_crime_count', 'grid_risk_score', 'grid_hour_risk_score', 'grid_day_risk_score', 'grid_time_period_risk_score', 'grid_weekend_risk_score']


In [19]:
df.to_parquet(
    "feature_engineered_crime_data.parquet",
    index=False
)

print("Feature-engineered dataset saved successfully!")
print("Shape:", df.shape)

Feature-engineered dataset saved successfully!
Shape: (3046607, 36)
